# 2026-06-30 dev_relative_pe_impl.ipynb
This is a dev notebook to inspect attention matrices and the implementation of relatiev PE  cia masking of gemma 

In [1]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.insert(0, "../src")
sys.path.insert(0, "../SPINE/src")


In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES']='1'

In [3]:
from prism.models import architectures
from transformers import AutoModelForCausalLM
import torch
from prism.training.train_v3 import _ensure_pad_tokens

/home/jporras/miniconda3/envs/GREP-PRISM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import transformers
transformers.__version__

'5.12.1'

## Setup

In [5]:
DEVICE='cuda'

In [6]:
import datasets

from prism.data import data
from transformers import AutoTokenizer

TRAIN_FILE = "../data/revised/gen/nav100_n30_gemma_data/split/formatted_all_new__train.json"
ARCHITECTURE = "graph_mask_llm"
TEXT_EDGE_LIST = "present"
N_SAMPLES = 1

MODEL_TAG = "google/gemma-4-12B-it"
tokenizer = AutoTokenizer.from_pretrained(MODEL_TAG, use_fast=True)

raw = datasets.load_dataset("json", data_files=[TRAIN_FILE], split="train")
train_dataset = data.preprocess_dataset(
    raw.select(range(N_SAMPLES)),
    tokenizer,
    architecture=ARCHITECTURE,
    text_edge_list=TEXT_EDGE_LIST,
)
collator = data.SpineDataCollator(tokenizer, mlm=False)
example = train_dataset[0]
batch = collator([example])

print(f"loaded {len(raw)} examples, using {N_SAMPLES}")
print(f"seq_len={len(example['input_ids'])}, pyg_nodes={batch['graphs'].num_nodes}")
print(f"batch keys: {sorted(batch.keys())}")

loaded 400 examples, using 1
seq_len=1367, pyg_nodes=31
batch keys: ['answer_node_idx', 'assistant_idx', 'attention_mask', 'edge_list_idx', 'graphs', 'injection_maps', 'input_ids', 'labels', 'scene_node_idx']


In [7]:
llm = AutoModelForCausalLM.from_pretrained(MODEL_TAG).to(DEVICE)
_text_hidden = llm.config.get_text_config().hidden_size

Loading weights: 100%|██████████| 677/677 [00:00<00:00, 5394.81it/s]


In [8]:
_ensure_pad_tokens(tokenizer, llm)

device = next(llm.parameters()).device
fwd = {
    "input_ids": batch["input_ids"].to(device),
    "attention_mask": batch["attention_mask"].to(device),
    "labels": batch["labels"].to(device),
    "graphs": batch["graphs"].to(device),
    "injection_maps": batch["injection_maps"],
}

print(type(llm).__name__, "hidden=", llm.config.get_text_config().hidden_size)


Gemma4UnifiedForConditionalGeneration hidden= 3840


In [9]:
print(fwd['input_ids'].shape)
print(fwd['attention_mask'].shape)
print(fwd['labels'].shape)
#print(fwd['graphs'].shape)
#print(fwd['injection_maps'].shape)

torch.Size([1, 1367])
torch.Size([1, 1367])
torch.Size([1, 1367])


In [10]:
# 0: [(383, 389), (1078, 1084), (1197, 1203), (1227, 1233), (1327, 1333)],
# node ID => [start token ID, endf token ID]
fwd['injection_maps'][0]

{0: [(383, 389), (1078, 1084), (1197, 1203), (1227, 1233), (1327, 1333)],
 2: [(396, 402), (1107, 1113)],
 25: [(341, 347),
  (759, 765),
  (816, 822),
  (863, 869),
  (973, 979),
  (992, 998),
  (999, 1005)],
 29: [(366, 372), (670, 676), (954, 960), (1054, 1060)],
 1: [(390, 395), (1095, 1100)],
 9: [(265, 270),
  (436, 441),
  (514, 519),
  (528, 533),
  (538, 543),
  (1085, 1090),
  (1191, 1196),
  (1236, 1241),
  (1273, 1278),
  (1297, 1302),
  (1337, 1342),
  (1359, 1364)],
 16: [(295, 300),
  (614, 619),
  (677, 682),
  (689, 694),
  (699, 704),
  (1101, 1106)],
 17: [(301, 306),
  (580, 585),
  (683, 688),
  (715, 720),
  (721, 726),
  (731, 736),
  (743, 748),
  (753, 758)],
 21: [(319, 324),
  (652, 657),
  (705, 710),
  (737, 742),
  (774, 779),
  (806, 811),
  (837, 842),
  (1126, 1131),
  (1136, 1141),
  (1146, 1151)],
 23: [(329, 334),
  (556, 561),
  (857, 862),
  (870, 875),
  (882, 887),
  (894, 899),
  (906, 911)],
 24: [(335, 340),
  (851, 856),
  (918, 923),
  (924,

# Modifying attention
1. Create a wrapper `class GraphMaskLLM(PreTrainedModel):`, implement `_install_graph_mask`
2. Use `AttentionInterface.register(_PRISM_PE_IMPL, _prism_pe_attention_forward)` to inject a customized forward pass. 


In [11]:
(llm.get_decoder().layers[1].self_attn.config) is (llm.get_decoder().layers[0].self_attn.config)

True

In [30]:
# reference output before modifying the llm:
with torch.no_grad():
    ref_out = llm(**fwd,output_hidden_states=True,output_attentions=True).copy()

ref_out.logits[:5],ref_out.logits.shape #torch.Size([1, seq len, Vocab size])

(tensor([[[  2.3594,   7.5625,   5.0938,  ...,  -8.3750,  -8.5000,  -8.5000],
          [  2.3438,   7.6250,   5.0938,  ...,  -8.3750,  -8.5000,  -8.5000],
          [  2.5000,   7.6250,   5.0938,  ...,  -8.3750,  -8.4375,  -8.4375],
          ...,
          [-23.8750,   1.6953,   0.1982,  ..., -15.0000, -15.0000, -15.0000],
          [-28.1250,   7.8438,   0.9102,  ..., -17.5000, -17.3750, -17.3750],
          [-27.5000,  10.8750,  -1.2188,  ..., -16.6250, -16.6250, -16.7500]]],
        device='cuda:0', dtype=torch.bfloat16),
 torch.Size([1, 1367, 262144]))

In [33]:
len(ref_out.attentions)

48

In [35]:
ref_out.attentions[0].shape

torch.Size([1, 16, 1367, 1367])

In [23]:
# from copy import copy
# backup_config = copy(llm.config)


In [24]:

# first_attn = llm.get_decoder().layers[0].self_attn
# # add a property to the `self_attn` config attribute to proecss graph attn. 
# # Note: config is shared across layers.

# llm.config._attn_implementation = 'test_attn'
# llm.config.__


In [ ]:
backup_config._attn_implementation

'eager'

In [29]:
# Example_data_for_custom_pe
len(ref_out.hidden_states)

49

In [36]:
ref_out.hidden_states[0].shape

torch.Size([1, 1367, 3840])

In [45]:
# Let's start where it matters. 
from torch.nn import functional as F
def _custom_pe_attention_forward(module, query, key, value, attention_mask,
                                scaling=None, dropout=0.0, **kwargs):
        return F.scaled_dot_product_attention(query, key, value, attention_mask)
    

query,key,value = ref_out.hidden_states[0],ref_out.hidden_states[0],ref_out.hidden_states[0]
query.shape
_custom_pe_attention_forward(None,query,key,value,None,None,0.0)


tensor([[[ 0.3086,  0.0515, 12.3750,  ..., -0.0605, -0.2246,  0.0591],
         [ 0.7617, -0.0498, -1.0625,  ...,  1.0938,  0.2051,  2.0000],
         [ 0.3262, -1.3516, -1.1406,  ...,  0.8906,  0.1230, -1.8203],
         ...,
         [ 0.0393, -0.6328,  0.5234,  ..., -0.0605,  0.0771, -0.5000],
         [-0.5586,  1.8203, -0.5195,  ...,  2.2812, -0.0179, -0.8984],
         [-0.5469, -0.1572, -0.3672,  ...,  0.2480, -0.2773, -0.0542]]],
       device='cuda:0', dtype=torch.bfloat16)

In [ ]:
from transformers import PreTrainedModel



AttentionInterface.register("test_attn", _custom_pe_attention_forward)

class GraphMaskLLM(PreTrainedModel):
    def __init__(self, llm):
        config = llm.config
        config._attn_implementation = 'eager'
        super().__init__(config)
        self.llm = llm
        self._install_graph_mask()

    def _install_graph_mask(self):
        print(self._decoder_layers())

    def forward(
        self,
        input_ids,
        attention_mask,
        labels,
        graphs,
        injection_maps,
        **kwargs,
    ):
        return self.llm(
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            labels=labels, 
            **kwargs)

augmented_model = GraphMaskLLM(llm)
with torch.no_grad():
    outputs = augmented_model(**fwd)

outputs

Gemma4UnifiedCausalLMOutputWithPast(loss=tensor(2.9364, device='cuda:0'), logits=tensor([[[  2.3594,   7.5625,   5.0938,  ...,  -8.3750,  -8.5000,  -8.5000],
         [  2.3438,   7.6250,   5.0938,  ...,  -8.3750,  -8.5000,  -8.5000],
         [  2.5000,   7.6250,   5.0938,  ...,  -8.3750,  -8.4375,  -8.4375],
         ...,
         [-23.8750,   1.6953,   0.1982,  ..., -15.0000, -15.0000, -15.0000],
         [-28.1250,   7.8438,   0.9102,  ..., -17.5000, -17.3750, -17.3750],
         [-27.5000,  10.8750,  -1.2188,  ..., -16.6250, -16.6250, -16.7500]]],
       device='cuda:0', dtype=torch.bfloat16), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, Dynamic